In [ ]:
# ── Cell 1: Colab GPU Setup ───────────────────────────────────────────────────
# Requires T4 GPU runtime: Runtime → Change runtime type → T4 GPU

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/features', exist_ok=True)
sys.path.insert(0, f'{PROJECT_DIR}/src')

# Install dependencies
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'sentence-transformers', 'datasets',
    'accelerate', 'scikit-learn', 'pandas', 'numpy', 'tqdm', 'scipy'
], check=True)

# Verify GPU
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. Runtime → Change runtime type → T4 GPU'
)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PROJECT_DIR = {PROJECT_DIR}')

In [ ]:
# ── Cell 2: Load Data ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')

from utils.data_loader import load_combined_data, records_to_texts_labels
import numpy as np

# Load from augmented data if available, else raw
from pathlib import Path
aug_path = Path(PROJECT_DIR) / 'data' / 'augmented' / 'augmented_train.json'
if aug_path.exists():
    import json as _json
    with open(aug_path) as f:
        records = _json.load(f)
    texts  = [r['text']  for r in records]
    labels = [int(r['label']) for r in records]
    print(f'Loaded augmented data: {len(texts)} samples')
else:
    records = load_combined_data(PROJECT_DIR)
    texts, labels = records_to_texts_labels(records)

labels = np.array(labels, dtype=np.int32)
print(f'Texts: {len(texts)}, Labels: {labels.shape}, Balance: {labels.mean():.2%} true')
print('Sample text:', texts[0][:200])


In [ ]:
# ── Cell 3: Extract NLI Internal-Consistency Features ────────────────────────
# Model : cross-encoder/nli-deberta-v3-large
# Output: (N, 5) array — contradiction_ratio, max_contradiction_score,
#         entailment_ratio, coherence_score, weighted_contradiction
# Est.  : ~2–3 hrs on T4

from features.tier1_nli import extract_nli_feature_matrix

print('Extracting NLI features (this takes ~2–3 hrs on T4) ...')
nli_features = extract_nli_feature_matrix(
    texts=texts,
    device='cuda',
    batch_size=32,
    checkpoint_path=f'{PROJECT_DIR}/features/_nli_checkpoint.npy',
)

print(f'NLI features shape : {nli_features.shape}')   # (N, 5)
print(f'Sample row         : {nli_features[0]}')

# Save checkpoint immediately
import numpy as np
np.save(f'{PROJECT_DIR}/features/_nli_features.npy', nli_features)
print('NLI features checkpoint saved.')

In [ ]:
# ── Cell 4: Extract FinBERT CLS Embeddings + Distance Features ───────────────
# Model : ProsusAI/finbert
# Output: (N, 67) array — 64-dim PCA projection + cosine_dist + mahal_dist
#                          + raw_norm (3 distance-based scalars)
# Est.  : ~10 mins on T4

from features.tier1_embeddings import extract_embedding_features

print('Extracting FinBERT CLS embedding features ...')
emb_features = extract_embedding_features(
    texts=texts,
    labels=labels,
    device='cuda',
    pca_components=64,
    batch_size=32,
)

print(f'Embedding features shape : {emb_features.shape}')  # (N, 66 or 67)
print(f'Sample row               : {emb_features[0, :5]}  ...')

import numpy as np
np.save(f'{PROJECT_DIR}/features/_emb_features.npy', emb_features)
print('Embedding features checkpoint saved.')

In [ ]:
# ── Cell 5: Extract Discourse Coherence Features ─────────────────────────────
# Model : sentence-transformers/all-mpnet-base-v2
# Output: (N, 4) array — consecutive_similarity_min, consecutive_similarity_std,
#                         first_last_similarity, mean_coherence
# Est.  : ~5 mins on T4

from features.tier1_coherence import extract_coherence_feature_matrix

print('Extracting discourse coherence features ...')
coherence_features = extract_coherence_feature_matrix(
    texts=texts,
    device='cuda',
    batch_size=64,
)

print(f'Coherence features shape : {coherence_features.shape}')  # (N, 4)
print(f'Sample row               : {coherence_features[0]}')

import numpy as np
np.save(f'{PROJECT_DIR}/features/_coherence_features.npy', coherence_features)
print('Coherence features checkpoint saved.')

In [ ]:
# ── Cell 6: Extract Epistemic Calibration Features (CPU) ─────────────────────
# Rule-based — no GPU needed
# Output: (N, 3) array — certainty_ratio, hedge_density,
#                         certainty_evidence_mismatch

from features.tier1_epistemic import extract_epistemic_feature_matrix

print('Extracting epistemic calibration features (CPU, rule-based) ...')
epistemic_features = extract_epistemic_feature_matrix(texts=texts)

print(f'Epistemic features shape : {epistemic_features.shape}')  # (N, 3)
print(f'Sample row               : {epistemic_features[0]}')

import numpy as np
np.save(f'{PROJECT_DIR}/features/_epistemic_features.npy', epistemic_features)
print('Epistemic features checkpoint saved.')

In [ ]:
# ── Cell 7: Concatenate All Tier 1 Features → Save ───────────────────────────

import numpy as np

# Load from checkpoints if cells were run in separate sessions
nli_features       = np.load(f'{PROJECT_DIR}/features/_nli_features.npy')
emb_features       = np.load(f'{PROJECT_DIR}/features/_emb_features.npy')
coherence_features = np.load(f'{PROJECT_DIR}/features/_coherence_features.npy')
epistemic_features = np.load(f'{PROJECT_DIR}/features/_epistemic_features.npy')

print('Individual feature shapes:')
print(f'  NLI        : {nli_features.shape}')
print(f'  Embeddings : {emb_features.shape}')
print(f'  Coherence  : {coherence_features.shape}')
print(f'  Epistemic  : {epistemic_features.shape}')

# Concatenate horizontally
tier1_features = np.hstack([
    nli_features,
    emb_features,
    coherence_features,
    epistemic_features,
])

print(f'\nTier 1 combined shape : {tier1_features.shape}')

# Save final arrays
np.save(f'{PROJECT_DIR}/features/tier1_features.npy', tier1_features)
np.save(f'{PROJECT_DIR}/features/labels.npy', labels)

print('✅ Saved:')
print(f'  {PROJECT_DIR}/features/tier1_features.npy  — shape {tier1_features.shape}')
print(f'  {PROJECT_DIR}/features/labels.npy           — shape {labels.shape}')

# Quick sanity check: are NaN-free?
assert not np.isnan(tier1_features).any(), 'NaN values detected in tier1_features!'
print('NaN check passed.')